In [0]:
dbutils.widgets.text("input_bcn2acn_parquet_file", "/Volumes/7_outgoing/pharos/accession_numbers/pharos_20260828_bcn_pid_acn.parquet", "parquet file generated by BCN to AccessionNbr notebook")
dbutils.widgets.text("output_full_table_name", "7_outgoing.pharos.pharos_20260828_imaging_report", "output table name")

In [0]:
input_bcn2acn_parquet_file = dbutils.widgets.get("input_bcn2acn_parquet_file")
bcn2acn_df = spark.read.parquet(input_bcn2acn_parquet_file)
display(bcn2acn_df.limit(100))

In [0]:
distinct_accession_df = bcn2acn_df.select("requested_accession_number").distinct()

imaging_report_df = spark.table("4_prod.pacs.imaging_report").select("AccessionNbr", "ReportEventId", "ReportEncntrId", "ExamCode")
blobdataset_df = spark.table("4_prod.rde.rde_blobdataset").select("EventID", "EventDesc", "AnonymizedText", "BlobStatus", "ClinicalSignificantDate")

joined_df = (
    distinct_accession_df
    .join(imaging_report_df, distinct_accession_df.requested_accession_number == imaging_report_df.AccessionNbr, "left")
    .join(blobdataset_df, imaging_report_df.ReportEventId == blobdataset_df.EventID, "left")
).drop("EventID")

display(joined_df.limit(10))

In [0]:
output_full_table_name = dbutils.widgets.get("output_full_table_name")
joined_df.write.mode("overwrite").saveAsTable(output_full_table_name)